In [1]:
#import glob
#import os

# Path to the folder containing the Excel files
#folder_path = "cleaned"

# Load all Excel files in the folder
#excel_files = glob.glob(os.path.join(folder_path, "*.xlsx"))

# Initialize an empty list to store the DataFrames
#df_list = []

# Loop through each file and read the Excel sheet into a DataFrame
#for file in excel_files:
    #df = pd.read_excel(file)
    #df_list.append(df)

# Combine all DataFrames by appending them
#combined_df = pd.concat(df_list, ignore_index=True)

# Display the combined DataFrame
#print(combined_df)

# Optionally, save the combined data to a new Excel file
#combined_df.to_csv("combined_property_data.csv", index=False)

In [2]:
# !pip uninstall numpy

In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string

In [4]:
# Download NLTK data files (run this once if not downloaded)
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Leibniz\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Leibniz\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [5]:
# Load data
property_df = pd.read_csv('QGIS_df.csv')  # property DataFrame
RR_CR_dct = {"condo": "RR", "house": "RR", "apartment":"RR", "commercial":"CR", "land":"CR"}
property_df["Property Type"] = property_df["Category"].map(RR_CR_dct)

zv_df = pd.read_csv('combined_property_data.csv') # Zonal Values DataFrame
zv_df[['Street', 'Vicinity', 'Barangay', 'City', 'Province']] = zv_df[['Street', 'Vicinity', 'Barangay', 'City', 'Province']].fillna('')

# Set up stop words and punctuation for preprocessing
stop_words = set(stopwords.words('english'))
punctuation = set(string.punctuation)

C:\Users\Leibniz\AppData\Local\Temp\ipykernel_1144\1364728511.py:6: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  zv_df = pd.read_csv('combined_property_data.csv') # Zonal Values DataFrame


In [6]:
# Define the classification function for Property Type in property_df
def classify_property_type(category):
    category = category.lower()
    if 'house' in category or 'condo' in category:
        return 'RR'
    elif 'commercial' in category or 'land' in category:
        return 'CC'
    return 'Unknown'  # In case there are other categories

# Apply the classification function to create Property Type column in property_df
property_df['Property Type'] = property_df['Category'].apply(classify_property_type)

In [7]:
# Function to preprocess location text (remove unwanted characters, tokenize)
def preprocess_location(location):
    # Tokenize and clean text: remove special characters and stop words
    tokens = word_tokenize(location.lower())
    cleaned_tokens = [token for token in tokens if token not in stop_words and token not in punctuation]
    cleaned_text = ' '.join(cleaned_tokens)
    return cleaned_text

# Function to preprocess location text (remove unwanted characters, tokenize)
def preprocess_location(location):
    tokens = word_tokenize(location.lower())
    cleaned_tokens = [token for token in tokens if token not in stop_words and token not in punctuation]
    cleaned_text = ' '.join(cleaned_tokens)
    return cleaned_text

In [ ]:
# Preprocess the locations
property_df['Processed_Location'] = property_df['Location'].apply(preprocess_location)
zv_df['Processed_Location'] = (zv_df['Street'] + " " + zv_df['Vicinity'] + " " + 
                               zv_df['Barangay'] + " " + zv_df['City'] + " " + zv_df['Province']).apply(preprocess_location)


# Cosine Match

In [ ]:
# Create a TF-IDF Vectorizer
vectorizer = TfidfVectorizer()

# Combine both property and zonal value locations
combined_locations = pd.concat([property_df['Processed_Location'], zv_df['Processed_Location']], ignore_index=True)

In [ ]:
# Fit and transform the combined locations
tfidf_matrix = vectorizer.fit_transform(combined_locations)

# Split the TF-IDF matrix for property and zonal value locations
property_tfidf = tfidf_matrix[:len(property_df)]
zv_tfidf = tfidf_matrix[len(property_df):]

In [ ]:
# Calculate cosine similarity between property locations and zonal value locations
cosine_similarities = cosine_similarity(property_tfidf, zv_tfidf)

# Assign the most similar zonal value based on both classification and similarity match
def get_best_zonal_match(similarities, zv_df, prop_classification):
    # Filter zonal values based on matching Property Type and Classification
    eligible_zonal_values = property_df[property_df['Property Type'] == prop_classification]
    if eligible_zonal_values.empty:
        return None  # No match found if no matching classifications
    
    # Use similarity scores only for the eligible zonal values
    eligible_similarities = similarities[eligible_zonal_values.index]
    best_match_idx = eligible_similarities.argmax()
    return eligible_zonal_values.iloc[best_match_idx]['Zonal Value']

MemoryError: Unable to allocate 43.3 GiB for an array with shape (72019, 80666) and data type float64

In [ ]:
# Apply the matching process
property_df['Matched_Zonal_Value'] = [
    get_best_zonal_match(cosine_similarities[i], zv_df, row['Classification'])
    for i, row in property_df.iterrows()
]

In [ ]:
# Show the results
print(property_df[['Location', 'Classification', 'Matched_Zonal_Value']])

# Fuzzy Match

In [25]:
from fuzzywuzzy import process

unique_property_locations = property_df['Processed_Location'].dropna().unique()
zonal_cities = zv_df['Processed_Location'].dropna().unique()

# Function to find best match using fuzzy matching
def fuzzy_match(location, choices, threshold=80):
    result = process.extractOne(location, choices)  # Get best match
    if result:  # Check if a match was found
        match, score = result
        return match if score >= threshold else None
    return None  # Return None if no match found

# Perform fuzzy matching only on unique locations
matched_dict = {loc: fuzzy_match(loc, zonal_cities) for loc in unique_property_locations}
# matched_dict.update({"Quezon City": 'Quezon (MM)', 'H-2, Dasmariñas': 'Dasmarinas'}) #Manually updated

# Map the matched results back to the original DataFrame
property_df["Matched_City_Zone"] = property_df["Processed_Location"].map(matched_dict)

# Show results
property_df[['Location', 'Matched_City_Zone']].head()

KeyboardInterrupt: 